# **TASK** **1**

# IMPORTING FILES


*   appointments.csv
*   clinics.csv


*   followup_notes.csv





In [9]:
import pandas as pd

url = "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/5e122459-7975-4534-957f-8397b0f9c03e/ecWvPsdV2FNmRZAa.csv"
df = pd.read_csv(url)
df.head()

,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


In [10]:
url2 = "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/0fe7e34d-9470-47d8-b7b1-d54b77e009fe/jbFzPdd2gB2Ex2TY.csv"
df2 = pd.read_csv(url2)
df2.head()

,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [8]:
url3 = "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/c1a9c6a2-5967-41fb-9133-cb6f78008699/qDCCmpyfss1kBJJB.csv"
df3 = pd.read_csv(url3)
df3.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


# **PART 1**



**TASK 1.1**


In [13]:
import json
import math

accepted_rows = []
rejected_rows = []

# Convert the DataFrame df3 to a list of dictionaries for row-by-row processing
all_rows = df3.to_dict(orient="records")
print(f"Total rows in DataFrame: {len(all_rows)}")

# Function to convert data types to integers and floats
def parse_row(row):
    parsed = row.copy()

    # If values are float NaNs from pandas, float() or int() might fail
    # We need to handle them carefully
    if isinstance(row["age"], float) and math.isnan(row["age"]):
        raise ValueError("Missing age")
    if isinstance(row["fee_inr"], float) and math.isnan(row["fee_inr"]):
        raise ValueError("Missing fee_inr")
    if isinstance(row["prior_no_shows"], float) and math.isnan(row["prior_no_shows"]):
        raise ValueError("Missing prior_no_shows")
    if isinstance(row["distance_km"], float) and math.isnan(row["distance_km"]):
        raise ValueError("Missing distance_km")

    parsed["age"] = int(float(row["age"]))
    parsed["fee_inr"] = int(float(row["fee_inr"]))
    parsed["prior_no_shows"] = int(float(row["prior_no_shows"]))
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

# Loop through every row and check for errors
for row in all_rows:
    try:
        converted_row = parse_row(row)
        accepted_rows.append(converted_row)
    except (ValueError, TypeError):
        # Handle NaNs and parsing errors, then save to rejected list
        # Convert NaN values to None so that the output is valid JSON
        cleaned_row = {k: (None if isinstance(v, float) and math.isnan(v) else v) for k, v in row.items()}
        rejected_rows.append(cleaned_row)

print(f"Accepted rows: {len(accepted_rows)}")
print(f"Rejected rows: {len(rejected_rows)}")

# Save the rejected rows into a JSON file
with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected_rows, f, indent=2)

print("Task 1.1 complete! rejected_rows.json has been created.")

Total rows in DataFrame: 1020
Accepted rows: 988
Rejected rows: 32
Task 1.1 complete! rejected_rows.json has been created.


**Task 1.2 — Inspect and clean, pass**

In [16]:
import pandas as pd

# Use the already loaded DataFrame df3
df = df3.copy()

# 1. Print initial inspections
print("--- Initial Inspection ---")
print("Shape:", df.shape)
print("\n--- Info ---")
df.info()
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\nDuplicates count:", df.duplicated().sum())

# (a) Drop exact duplicate rows
df.drop_duplicates(inplace=True)
print(f"Row count after dropping duplicates: {len(df)}")

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

# (c) Convert appt_date and booked_on to datetime objects
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], errors="raise")

# (d) Drop rows where appt_date or status is missing
df.dropna(subset=["appt_date", "status"], inplace=True)
print(f"Row count after dropping missing appt_date or status: {len(df)}")

--- Initial Inspection ---
Shape: (1020, 14)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

--- Missing Values ---
appointme

**Task 1.3 — Clean, pass 2: validity, gaps and outliers**

In [17]:
import numpy as np

print("--- TASK 1.3: Clean Pass 2 ---")

# (a) Treat any age below 1 or above 100 as invalid and set it to missing (NaN)
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
num_invalid_ages = invalid_age_mask.sum()
df.loc[invalid_age_mask, "age"] = np.nan
print(f"Invalid ages set to missing: {num_invalid_ages}")

# (b) Fill missing age with the median (rounded to integer) and missing distance_km with its median
median_age = df["age"].median()
median_distance = df["distance_km"].median()

df["age"].fillna(median_age, inplace=True)
df["age"] = df["age"].round().astype(int)

df["distance_km"].fillna(median_distance, inplace=True)

# (c) Compute Q1, Q3, IQR, and cap outliers using np.clip
def cap_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    # Count how many values fall outside the fences
    num_capped = (series < lower_fence).sum() + (series > upper_fence).sum()
    capped = np.clip(series, lower_fence, upper_fence)
    return capped, num_capped

df["distance_km"], dist_capped_count = cap_outliers(df["distance_km"])
df["fee_inr"], fee_capped_count = cap_outliers(df["fee_inr"])

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print(f"Distance values capped: {dist_capped_count}")
print(f"Fee values capped: {fee_capped_count}")

--- TASK 1.3: Clean Pass 2 ---
Invalid ages set to missing: 5
Distance values capped: 26
Fee values capped: 3


/tmp/ipykernel_716/1129417373.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["age"].fillna(median_age, inplace=True)
/tmp/ipykernel_716/1129417373.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

**Task 1.4 — Features and a NumPy summary**

In [18]:
import numpy as np

# 1. Create lead_days and no_show features
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = df["status"].apply(lambda x: 1 if x == "No-Show" else 0)

# 2. Convert selected columns to a NumPy array X
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Compute the mean of each column (axis=0) without loops
col_means = np.mean(X, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# (b) Number and share of rows with lead_days > 14 using a boolean mask
lead_mask = X[:, 3] > 14
num_lead_gt_14 = lead_mask.sum()
share_lead_gt_14 = lead_mask.mean() * 100
print(f"Rows with lead_days > 14: {num_lead_gt_14} ({share_lead_gt_14:.1f}%)")

# (c) Z-scores of distance_km column and count of rows with |z| > 2
dist_col = X[:, 1]
z_scores = (dist_col - np.mean(dist_col)) / np.std(dist_col)
z_mask = np.abs(z_scores) > 2
num_z_gt_2 = z_mask.sum()
print(f"Rows with distance |z| > 2: {num_z_gt_2}")

Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: 296 (29.9%)
Rows with distance |z| > 2: 53


**Task 1.5 — Enrich the clinics from a live API**

In [21]:
import requests
import pandas as pd

# 1. Use the already loaded DataFrame df2
clinics_df = df2.copy()
temperatures = []

# 2. Loop through each clinic and call the Open-Meteo API
for idx, row in clinics_df.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    name = row["clinic_name"]
    temp_c = None

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Retry logic: attempt up to 2 times
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)

            # Check if status code is 200
            if response.status_code == 200:
                data = response.json()

                # Extract the first value of hourly temperature_2m
                if "hourly" in data and "temperature_2m" in data["hourly"]:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    break
                else:
                    print(f"Clinic '{name}': 'hourly' or 'temperature_2m' missing in JSON response.")
            else:
                print(f"Clinic '{name}': Non-200 status code received ({response.status_code}).")

        except requests.exceptions.RequestException as e:
            if attempt == 1:
                print(f"Clinic '{name}' failed after retry. Reason: {e}")

    # If it still fails, store None and carry on
    if temp_c is None:
        print(f"Clinic '{name}': Storing None as temp_c.")

    temperatures.append(temp_c)

# 3. Add temp_c column to the DataFrame and save to clinics_enriched.csv
clinics_df["temp_c"] = temperatures
clinics_df.to_csv("clinics_enriched.csv", index=False)

print("clinics_enriched.csv saved successfully with 8 rows!")

clinics_enriched.csv saved successfully with 8 rows!


**Task 1.6 — Export the clean table and write the SQL load script**

In [25]:
# 1. Sort by appointment_id and format dates as YYYY-MM-DD text strings
df.sort_values(by="appointment_id", inplace=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# 2. Select the exact 16 columns in the required order
columns_order = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"
]
appointments_clean = df[columns_order]

# 3. Validation checks: check uniqueness of appointment_id and zero missing values
assert appointments_clean["appointment_id"].is_unique, "appointment_id is not unique!"
assert appointments_clean.isnull().sum().sum() == 0, "Missing values found in final export!"
print("Validation passed: appointment_id is unique and 0 missing values found.")

# 4. Save the cleaned table to CSV
appointments_clean.to_csv("appointments_clean.csv", index=False)
print("appointments_clean.csv saved successfully!")

# 5. Generate load_data.sql using a loop and f-strings
sql_statements = []

# (a) Write all clinic statements FIRST (due to foreign key dependencies)
for _, row in clinics_df.iterrows():
    temp_val = row["temp_c"]
    temp_sql = "NULL" if pd.isna(temp_val) else str(temp_val)
    # Fix: use 'clinic_id' instead of 'id', and 'clinic_name' instead of 'name'
    stmt = f"INSERT INTO clinics VALUES ('{row['clinic_id']}', '{row['clinic_name']}', '{row['city']}', '{row['tier']}', {row['latitude']}, {row['longitude']}, {temp_sql});"
    sql_statements.append(stmt)

# (b) Write all appointment statements next
for _, row in appointments_clean.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"'{row['appointment_id']}', '{row['patient_id']}', '{row['clinic_id']}', "
        f"'{row['booked_on']}', '{row['appt_date']}', {row['age']}, '{row['gender']}', "
        f"{row['distance_km']}, {row['fee_inr']}, '{row['booking_channel']}', "
        f"'{row['visit_type']}', '{row['reminder_sent']}', {row['prior_no_shows']}, "
        f"'{row['status']}', {row['lead_days']}, {row['no_show']});"
    )
    sql_statements.append(stmt)

# 6. Save everything to load_data.sql
with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"load_data.sql saved successfully with {len(sql_statements)} statements (8 clinics + 990 appointments)!")

Validation passed: appointment_id is unique and 0 missing values found.
appointments_clean.csv saved successfully!
load_data.sql saved successfully with 998 statements (8 clinics + 990 appointments)!
